# PAN-Format Demo Identifier Generator (with Duplicate Prevention)

> **Important:** This notebook generates **synthetic, PAN-format demo identifiers only**. It does not issue, verify, or generate an official Indian PAN. The final letter below is random and is **not** an official PAN checksum. Use this only for programming practice/testing; never use generated values as real identity documents.

## Improvements
- Validates holder name and category.
- Builds a 10-character string in a PAN-like layout.
- Checks for duplicates **before recording/returning** a generated value.
- Stores identifiers in SQLite, so duplicate checking survives kernel restarts.
- Uses a database primary key as a second line of defense.


## 1. Imports and configuration

The SQLite database file is created in the notebook's current working directory. Keep it to preserve the registry between sessions.

In [ ]:
import random
import sqlite3
import string
from pathlib import Path

DB_PATH = Path("demo_pan_registry.sqlite3")

STATUS_CODES = {
    "individual": "P",
    "company": "C",
    "trust": "T",
}
ALPHABETS = string.ascii_uppercase
DIGITS = string.digits
MAX_ATTEMPTS = 10_000


## 2. Create a persistent registry

The identifier is a primary key, so SQLite rejects duplicates at the storage layer too.

In [ ]:
def get_connection():
    return sqlite3.connect(DB_PATH, timeout=10)


def initialize_registry():
    with get_connection() as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS issued_demo_pan (
                demo_pan TEXT PRIMARY KEY,
                holder_type TEXT NOT NULL,
                holder_name TEXT NOT NULL,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
            """
        )


initialize_registry()
print(f"Demo registry ready: {DB_PATH.resolve()}")


## 3. Input validation and candidate generation

This creates a **format-shaped demo value**, not an official PAN. The last letter is random rather than a real check digit.

In [ ]:
def normalize_holder_type(holder_type: str) -> str:
    if not isinstance(holder_type, str):
        raise TypeError("Holder type must be text.")
    key = holder_type.strip().lower()
    if key not in STATUS_CODES:
        allowed = ", ".join(STATUS_CODES)
        raise ValueError(f"Unsupported holder type. Choose one of: {allowed}.")
    return key


def validate_holder_name(holder_name: str) -> str:
    if not isinstance(holder_name, str):
        raise TypeError("Holder name must be text.")
    name = " ".join(holder_name.strip().split())
    if not name:
        raise ValueError("Holder name cannot be empty.")
    return name


def generate_candidate(holder_type: str, holder_name: str) -> str:
    category = normalize_holder_type(holder_type)
    name = validate_holder_name(holder_name)
    first_three = "".join(random.choices(ALPHABETS, k=3))
    fourth = STATUS_CODES[category]
    fifth = name.split()[-1][0].upper()
    four_digits = "".join(random.choices(DIGITS, k=4))
    demo_final_letter = random.choice(ALPHABETS)  # Not an official checksum.
    return f"{first_three}{fourth}{fifth}{four_digits}{demo_final_letter}"


## 4. Check duplicates before issuing/recording

The generator checks the registry first. If the candidate exists, it retries. The database primary key handles a possible race between checking and inserting.

In [ ]:
def is_already_issued(candidate: str) -> bool:
    with get_connection() as conn:
        row = conn.execute(
            "SELECT 1 FROM issued_demo_pan WHERE demo_pan = ?",
            (candidate.upper(),),
        ).fetchone()
    return row is not None


def generate_unique_demo_pan(holder_type: str, holder_name: str) -> str:
    category = normalize_holder_type(holder_type)
    name = validate_holder_name(holder_name)

    for _ in range(MAX_ATTEMPTS):
        candidate = generate_candidate(category, name)

        # Explicit duplicate check BEFORE recording/returning the identifier.
        if is_already_issued(candidate):
            continue

        try:
            with get_connection() as conn:
                conn.execute(
                    """
                    INSERT INTO issued_demo_pan (demo_pan, holder_type, holder_name)
                    VALUES (?, ?, ?)
                    """,
                    (candidate, category, name),
                )
            return candidate  # Return only after successful insert.
        except sqlite3.IntegrityError:
            # Another call may have inserted the same value after our SELECT.
            continue

    raise RuntimeError(
        "Could not find an unused demo identifier after many attempts. "
        "Check the registry or increase MAX_ATTEMPTS."
    )


## 5. Generate a demo identifier

Accepted categories: `individual`, `company`, and `trust`.

In [ ]:
holder_type = input("Holder type (individual/company/trust): ")
holder_name = input("Holder name: ")

try:
    demo_pan = generate_unique_demo_pan(holder_type, holder_name)
    print("Generated demo identifier:", demo_pan)
    print("Saved to the local demo registry.")
    print("Reminder: this is NOT an official or usable PAN.")
except (TypeError, ValueError, RuntimeError) as exc:
    print("Could not generate identifier:", exc)


## 6. View previously recorded demo identifiers

Use synthetic test data only; avoid storing real identity details in this practice database.

In [ ]:
def list_issued_demo_pans():
    with get_connection() as conn:
        rows = conn.execute(
            """
            SELECT demo_pan, holder_type, holder_name, created_at
            FROM issued_demo_pan
            ORDER BY created_at DESC, demo_pan
            """
        ).fetchall()
    return rows


for row in list_issued_demo_pans():
    print(row)


## 7. Quick duplicate-prevention test

The following test generates multiple synthetic values and checks that the returned values are unique.

In [ ]:
test_values = [
    generate_unique_demo_pan("individual", "Example Person")
    for _ in range(20)
]

assert len(test_values) == len(set(test_values)), "Duplicate demo identifier found!"
assert all(len(value) == 10 for value in test_values)
assert all(value.isalnum() and value.isupper() for value in test_values)

print(f"Passed: {len(test_values)} unique 10-character demo identifiers generated.")


## Notes

- Deleting `demo_pan_registry.sqlite3` deletes the local registry and duplicate history. Keep it for cross-session duplicate prevention.
- This notebook is not connected to the Income Tax Department and cannot issue or validate real PANs.
- For a production system, use an authorized official integration and protect personal data appropriately.